# HH NEURON Process Notebook - RFB Morphology

This scratch notebook stress-tests the notebook runtime with a fast HH point-cell simulation and multiple subprocess-rendered line panels. The backend runs in a child process, morphology interaction uses the local RFB widget, and line rendering stays out of the notebook kernel.


In [ ]:
import os

# Hybrid notebook mode:
# - cnv.show(build_source) builds/runs the NEURON source in a child process
# - RFB keeps morphology camera interaction local to the notebook widget
# - render-process mode keeps the multi-panel trace renderer out of the kernel
os.environ["CNV_NOTEBOOK_RFB"] = "1"
os.environ["CNV_NOTEBOOK_RENDER_PROCESS"] = "1"
os.environ.setdefault("COMPNV_PERF_LOG", "scratch/.compneurovis/notebook-logs")
os.environ["CNV_NOTEBOOK_TRACE_DPI"] = "150"
os.environ["CNV_NOTEBOOK_TRACE_QUALITY"] = "110"

import compneurovis as cnv


In [2]:
DT = 0.0125
DISPLAY_DT = 0.05
TRACE_WINDOW_MS = 80.0
POINT_LENGTH_UM = 12.6157
POINT_DIAM_UM = 12.6157
CONTINUOUS_CLAMP_DUR_MS = 1.0e9


def build_source():
    from neuron import h
    import compneurovis as cnv

    params = {
        "clamp_amp": 0.1,
        "gnabar": 0.12,
        "gkbar": 0.036,
        "gl": 0.0003,
    }

    soma = h.Section(name="hh_soma")
    soma.L = POINT_LENGTH_UM
    soma.diam = POINT_DIAM_UM
    soma.nseg = 1
    soma.insert("hh")

    seg = soma(0.5)
    stim = h.IClamp(seg)
    stim.delay = 0.0
    stim.dur = CONTINUOUS_CLAMP_DUR_MS

    h.dt = DT
    h.celsius = 6.3

    def apply_parameters():
        for item in soma:
            item.gnabar_hh = float(params["gnabar"])
            item.gkbar_hh = float(params["gkbar"])
            item.gl_hh = float(params["gl"])
        stim.amp = float(params["clamp_amp"])

    def set_param(name, value):
        params[name] = float(value)
        apply_parameters()

    apply_parameters()
    h.finitialize(-65.0)

    src = cnv.neuron.source(
        sections=[soma],
        dt=DT,
        display_dt=DISPLAY_DT,
        flush_dt=0.5,
        v_init=-65.0,
        title="HH NEURON RFB process notebook",
    )

    src.morphology(
        variable="v",
        name="Voltage morphology",
        unit="mV",
        color_limits=(-80.0, 55.0),
        max_refresh_hz=20.0,
    )

    voltage_data = src.record_refs(
        "Voltage",
        refs=(seg._ref_v,),
        series=("Voltage",),
        unit="mV",
        window=TRACE_WINDOW_MS,
    )
    src.line(
        "Voltage",
        source=voltage_data,
        rolling_window=TRACE_WINDOW_MS,
        y_label="Voltage",
        y_unit="mV",
        y_min=-85.0,
        y_max=55.0,
        pen="#00d2be",
        max_refresh_hz=20.0,
    )

    current_data = src.record_refs(
        "Input current",
        refs=(stim._ref_i,),
        series=("Input current",),
        unit="nA",
        window=TRACE_WINDOW_MS,
    )
    src.line(
        "Input current",
        source=current_data,
        rolling_window=TRACE_WINDOW_MS,
        y_label="Current",
        y_unit="nA",
        y_min=-0.05,
        y_max=0.25,
        pen="#2356b8",
        max_refresh_hz=20.0,
    )

    gating_data = src.record_refs(
        "Gating",
        refs=(seg._ref_m_hh, seg._ref_h_hh, seg._ref_n_hh),
        series=("m", "h", "n"),
        window=TRACE_WINDOW_MS,
    )
    src.line(
        "Gating",
        source=gating_data,
        rolling_window=TRACE_WINDOW_MS,
        y_label="Gate value",
        y_min=-0.05,
        y_max=1.05,
        show_legend=True,
        series_colors={"m": "#ff8c00", "h": "#ff50b4", "n": "#7d3cff"},
        max_refresh_hz=20.0,
    )

    ionic_data = src.record_refs(
        "Ionic currents",
        refs=(seg._ref_ina, seg._ref_ik, seg._ref_il_hh),
        series=("I_Na", "I_K", "I_L"),
        unit="mA/cm^2",
        window=TRACE_WINDOW_MS,
    )
    src.line(
        "Ionic currents",
        source=ionic_data,
        rolling_window=TRACE_WINDOW_MS,
        y_label="Current density",
        y_unit="mA/cm^2",
        y_min=-5.0,
        y_max=5.0,
        show_legend=True,
        series_colors={"I_Na": "#ff8c00", "I_K": "#7d3cff", "I_L": "#00d2be"},
        max_refresh_hz=20.0,
    )

    def slider(name, label, minimum, maximum, steps):
        src.control(
            name,
            label=label,
            get=lambda name=name: params[name],
            set=lambda ctx, value, name=name: set_param(name, value),
            min=minimum,
            max=maximum,
            presentation=cnv.ControlPresentationSpec(kind="slider", steps=steps),
        )

    slider("clamp_amp", "IClamp amplitude (nA)", -0.20, 0.50, 280)
    slider("gnabar", "Na conductance gNa (S/cm^2)", 0.01, 0.30, 290)
    slider("gkbar", "K conductance gK (S/cm^2)", 0.005, 0.10, 190)
    slider("gl", "Leak conductance gL (S/cm^2)", 0.00005, 0.005, 220)
    src.action("reset", label="Reset", fn=lambda ctx: h.finitialize(-65.0), resets_fields=True)

    return src


In [3]:
widget = cnv.show(build_source)
widget


Qt's default DPI awareness context is DPI_AWARENESS_CONTEXT_PER_MONITOR_AWARE_V2. If you know what you are doing, you can overwrite this default using qt.conf (https://doc.qt.io/qt-6/highdpi.html#configuring-windows).


Morphology visual generated in 0.02s
